In [9]:
from phase_II.nifty_re_playground.strain_tools import *
%matplotlib tk
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.signal.windows import tukey

In [10]:
def mean_preserving_tukey(arr, alpha=0.2, axis=None):
    """
    Subtract mean along axis, apply Tukey window along axis, then add mean back.
    """
    arr = np.asarray(arr)
    mean = arr.mean(axis=axis, keepdims=True)
    arr_zero = arr - mean

    # shape along axis
    N = arr.shape[axis] if axis is not None else arr.size
    w = tukey(N, alpha=alpha)

    # reshape w for broadcasting along axis
    if axis is not None:
        shape = [1]*arr.ndim
        shape[axis] = N
        w = w.reshape(shape)

    arr_windowed = arr_zero * w + mean
    return arr_windowed


def tukey_window_matrix(mat, ax=None, alpha=0.2):
    """
    Apply a 2D Tukey window to a matrix in an intuitive way.

    Parameters
    ----------
    mat : ndarray
        2D array to window
    ax : {None, 0, 1}
        - None : window both axes
        - 0    : window rows individually (FFT along rows becomes periodic)
        - 1    : window columns individually (FFT along columns becomes periodic)
    alpha : float
        Tukey shape parameter (0 < alpha <= 1)

    Returns
    -------
    mat_windowed : ndarray
        Windowed matrix
    """
    mat = np.asarray(mat)
    ny, nx = mat.shape

    if ax is None:
        wx = tukey(nx, alpha=alpha)
        wy = tukey(ny, alpha=alpha)
        return mat * wy[:, None] * wx[None, :]

    elif ax == 0:
        # Window each row independently along its columns
        wx = tukey(nx, alpha=alpha)    # along row
        return mat * wx[None, :]       # broadcast along rows

    elif ax == 1:
        # Window each column independently along its rows
        wy = tukey(ny, alpha=alpha)    # along column
        return mat * wy[:, None]       # broadcast along columns

    else:
        raise ValueError("axis must be None, 0, or 1")


def tukey_window_array(array, alpha=0.2):
    wx = tukey(len(array), alpha=alpha)
    return array * wx

In [11]:

def boundary_differences(M):
    """
    M: 2D array (can be complex)
    returns:
        col_diff, row_diff  (complex arrays)
    """
    M = np.asarray(M)

    # columns: last row - first row
    col_diff = M[-1, :] - M[0, :]

    # rows: last column - first column
    row_diff = M[:, -1] - M[:, 0]

    return col_diff, row_diff


def plot_boundary_differences(M):
    col_diff, row_diff = boundary_differences(M)

    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=False)

    # columns
    axs[0].plot(col_diff.real, label="Re(col diff)")
    axs[0].plot(col_diff.imag, label="Im(col diff)")
    axs[0].set_title("Column boundary differences")
    axs[0].set_xlabel("Column index")
    axs[0].legend()

    # rows
    axs[1].plot(row_diff.real, label="Re(row diff)")
    axs[1].plot(row_diff.imag, label="Im(row diff)")
    axs[1].set_title("Row boundary differences")
    axs[1].set_xlabel("Row index")
    axs[1].legend()

    plt.tight_layout()
    plt.show()


def Stress_re_debug(xi, time, supress_print=False, downsample=False, norm="ortho", tukey_window_where_necessary=False):
    """
    Implements S_ft, i.e. rows are frequencies and columns are times.

    See also nifty8 `Stress` function.

    :param xi: jnp.array        A field to calculate the wigner function for. Either of complex or real data type.
                                If complex, assumed to be in DFT standard order (DC first, then positives then negatives).
    :param time: jnp.array      The real-space time array at which xi (or its iFFT if complex) was sampled at.
    :param supress_print: bool, Print imaginary part diagonstics (Wigner function should be real).
    :return:
    """

    t0 = time[0]
    dt = time[1]-time[0]
    N = len(xi)
    f = jnp.fft.fftfreq(N, d=dt)
    k = f.copy()
    df = f[1] - f[0]
    t = jnp.arange(N) / (N*df)  # dual time, equal to input time - time[0].
    T = N * dt

    FFT_physical = lambda x, ax=-1: jnp.fft.fft(x, norm=norm, axis=ax) * T / jnp.sqrt(N)
    iFFT_physical = lambda x, ax=-1: jnp.fft.ifft(x, norm=norm, axis=ax) * jnp.sqrt(N) / T

    if jnp.iscomplexobj(xi):
        print("INVERSE FOURIER TRANSFORMING xi")
        xi = iFFT_physical(xi)  # go to real space
    else:
        print("not INVERSE FOURIER TRANSFORMING xi")

    if downsample:
        step = 2
        xi = xi[::step]
        time = time[::step]

    if not supress_print:
        print("\nCalculating stress...")

    t_c = t[:, None]  # time cast
    k_c = k[None, :]  # shift frequencies cast
    xi_c = xi[:, None]  # xi values cast as rows

    if not supress_print:
        print("\t Calculating zeta plus")
    if tukey_window_where_necessary:
        plot_boundary_differences(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c)
        zeta_plus = tukey_window_matrix(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
        # plot_boundary_differences(zeta_plus)
    else:
        zeta_plus = jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta minus")

    if tukey_window_where_necessary:
        plot_boundary_differences(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c)
        zeta_minus = tukey_window_matrix(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
        # plot_boundary_differences(zeta_minus)
        # stop
    else:
        zeta_minus = jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c  # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta plus in Fourier space")
    tilde_zeta_plus = FFT_physical(zeta_plus, ax=0)

    if not supress_print:
        print("\t Calculating zeta minus in Fourier space")
    tilde_zeta_minus = FFT_physical(zeta_minus, ax=0)

    if not supress_print:
        print("\t Calculating Phi matrix")
    if tukey_window_where_necessary:
        plot_boundary_differences(tilde_zeta_plus * tilde_zeta_minus.conj())
        # Phi = tukey_window_matrix(tilde_zeta_plus * tilde_zeta_minus.conj(), ax=1)  # domain = (h_space, h_space)
        print("NOT TUKEYING PHI, since it looks like in all cases its almost periodic...")
        Phi = tilde_zeta_plus * tilde_zeta_minus.conj()  # domain = (h_space, h_space)
        # plot_boundary_differences(Phi)
        # stop
    else:
        Phi = tilde_zeta_plus * tilde_zeta_minus.conj()  # domain = (h_space, h_space)

    if not supress_print:
        print("\t Inverse Fourier-Transforming columns of Phi matrix")
    S = iFFT_physical(Phi, ax=1)
    S.block_until_ready()

    if not supress_print:
        print("\t ... Done")
    if not supress_print:
        diagnostic = jnp.abs(jnp.mean(S.imag))
        tmp = float(diagnostic)
        if diagnostic < 1e-10:
            print(f"\u2714 Mean imaginary part of stress field is smaller than 1e-10 threshold ({diagnostic}) ")
        else:
            raise_warning(
                f"Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 ({diagnostic}).")
    return S, t+t0, f

In [12]:
def generate_white_noise_stress_matrices(number_of_matrices, time_array, std=1, supress_print=False, complex=False, norm="ortho"):
    """

    :param number_of_matrices:  How many matrices to generate
    :param time_array:         The array containing the time samples
    :param std:                 Is multiplied onto xi iid. For example, if you want to represent a Dirac Delta, the covariance should be Kronecker Delta * T,
                                where T is the length of the time domain. Therefore, variables have to be generated that are scaled with the standard deviation sqrt(T)
    :param supress_print:
    :return:
    """
    S_mat_collection = []
    N = len(time_array)
    for i in range(number_of_matrices):
            print(f"Calcuting white noise stress matrix, iteration {i} out of {number_of_matrices}")
            if complex:
                 white_noise = 1/2*np.random.randn(N) + 1/2*np.random.randn(N) * 1j  # I have not calculated the variance or mean for the complex fields, therefore
                 # I cannot benchmark this
            else:
                white_noise = np.random.standard_normal(N)
            white_noise_scaled = std * white_noise
            stress, _, _ = Stress_re_debug(white_noise_scaled, time=time_array, supress_print=supress_print, norm=norm)
            S_mat_collection.append(np.array(stress))

    print("Done, wrapping result in numpy array")
    return np.array(S_mat_collection)

### White noise Wigner function for real and complex field at different resolutions; pure white noise matrix

In [13]:
res_1 = 256
res_2 = 256*10
T = 3
time_res1 = jnp.linspace(0, T, res_1)
time_res2 = jnp.linspace(0, T, res_2)

delta_t_1 = time_res1[1]-time_res1[0]
delta_t_2 = time_res2[1]-time_res2[0]

xi_white_complex_res1 = np.random.standard_normal(res_1) + 1j*np.random.standard_normal(res_1)

xi_white_real_res1 = np.random.standard_normal(res_1)
xi_white_real_res2 = np.random.standard_normal(res_2)


In [ ]:
white_stress_complex_res1, t_dual_res_1, f_dual_res_1 = Stress_re_debug(xi=xi_white_complex_res1, time=time_res1)

In [ ]:
white_stress_real_res1, t_dual_res_1, f_dual_res_1 = Stress_re_debug(xi=xi_white_real_res1, time=time_res1)

In [ ]:
white_stress_real_res2, t_dual_res_2, f_dual_res_2 = Stress_re_debug(xi=xi_white_real_res2, time=time_res2, tukey_window_where_necessary=True)

In [ ]:
pure_white_noise_res1 = np.random.standard_normal(white_stress_real_res1.shape)
pure_white_noise_res2 = np.random.standard_normal(white_stress_real_res2.shape)

In [ ]:
# Visualize
apply_smoothing=True
save_current_figure=False

# fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True)

# flattened_axs = axs.flatten()

# visualize_stress(white_stress_complex_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Wigner function: complex $\xi$, low resolution")
# visualize_stress(white_stress_real_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Wigner function: real $\xi$, low resolution")
visualize_stress(white_stress_real_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Wigner function: real $\xi$, high resolution")
# visualize_stress(pure_white_noise_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Pure white noise, low resolution")
# visualize_stress(pure_white_noise_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Pure white noise, smoothed", show=False)
# visualize_stress(pure_white_noise_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=not apply_smoothing, save_fig=save_current_figure, tl=r"Pure white noise, smoothed", show=False, yl=None)
plt.show()

In [ ]:
visualize_stress(white_stress_real_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=False, save_fig=False, tl=r"Wigner function: real $\xi$, high resolution")

In [ ]:
num_of_samples = 10
white_noise_stress_collection = generate_white_noise_stress_matrices(number_of_matrices=num_of_samples, time_array=time_res2, supress_print=True, std=1/np.sqrt(delta_t_2),
                                                                     complex=False, norm="ortho")

In [ ]:
white_noise_stress_average = np.mean(white_noise_stress_collection, axis=0)

In [ ]:
print(np.mean(white_noise_stress_average), "should be 1")
print(np.std(white_noise_stress_average), f"should be {jnp.sqrt(res_2)/jnp.sqrt(num_of_samples)} = sqrt(delta(0) in time * delta(0) in frequency) / sqrt(num_of_samples) ")

In [ ]:
visualize_stress(white_noise_stress_average, rows=f_dual_res_1, cols=t_dual_res_1
                 )

In [ ]:
res = res_2
place_to_place_differing_gaussian_matrix = np.zeros((res, res))
for i in range(res):
    for j in range(res):
        rnd_sigma = np.abs(2*np.random.standard_normal(1)+.2)
        # rnd_sigma = 2
        rnd_mean  = np.abs(2*np.random.standard_normal(1)+2)
        # rnd_mean  = 1
        gaussian_sample = rnd_mean + rnd_sigma*np.random.standard_normal(size=1)
        place_to_place_differing_gaussian_matrix[i, j] = gaussian_sample[0]

In [ ]:
res = res_2
place_to_place_differing_lognormal_matrix = np.zeros((res, res))
for i in range(res):
    for j in range(res):
        rnd_sigma = np.abs(2*np.random.standard_normal(1)+.2)
        rnd_mean  = np.abs(2*np.random.standard_normal(1)+2)
        ln_sample = np.random.lognormal(rnd_mean, rnd_sigma, size=1)[0]
        place_to_place_differing_lognormal_matrix[i, j] = ln_sample

In [ ]:
res = res_2
lognormal_matrix = np.zeros((res, res))
for i in range(res):
    for j in range(res):
        ln_sample = np.random.lognormal(mean=1, sigma=0.1, size=1)[0]
        lognormal_matrix[i, j] = ln_sample

In [ ]:
# visualize_stress(place_to_place_differing_gaussian_matrix, rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)
visualize_stress(pure_white_noise_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [ ]:
visualize_stress(np.log(place_to_place_differing_lognormal_matrix), rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [ ]:
visualize_stress(np.log(lognormal_matrix), rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [ ]:
plt.hist(np.sum(place_to_place_differing_gaussian_matrix.real, axis=0), bins=100)
plt.show()

In [ ]:
visualize_stress(white_stress_real_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=False)

## Numerical relativity template

In [6]:
from phase_I.utils.config_jupyter_notebooks import *
nrt_strain_values = jnp.array(np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19)
nrt_time_values = jnp.array(np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt") - zero_time)

Important variables: 
		signal_strip_time, signal_strip_strain 
		signal_strip_strain_tapered
		strain
		time_domain_strip
		N


In [ ]:
plt.plot(nrt_time_values, nrt_strain_values)
plt.show()

In [ ]:
print("come here")
# white_stress_real_res2, t_dual_res_2, f_dual_res_2 = Stress_re_debug(xi=xi_white_real_res2, time=time_res2, tukey_window_where_necessary=True)
S_mat_nrt_tukey, t_dual_nrt, f_nrt = Stress_re_debug(xi=nrt_strain_values, time=nrt_time_values, tukey_window_where_necessary=True)

In [ ]:
visualize_stress(S_mat_nrt_tukey, rows=f_nrt, cols=t_dual_nrt, smooth=False, tl="tukey windowed", )#xlim=(16.2,16.5), ylim=(-300,300))
# visualize_stress(white_stress_real_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [14]:
from phase_II.utils.helpers import whiten, bandpass

def get_xi_for_event(event_idx=3, off_center=1.5, abs_path_to_use=None):
    # Define events
    events = [
        {"gw_name": 'GW150914', "duration":32, "unpack":True, "version": 3, "sample_rate":4096, "center_at": 1126259462.4+off_center},
        {"gw_name": 'GW150914', "duration":4096, "unpack":True, "version": 4, "sample_rate":4096, "center_at": 1126259598-4+off_center},
        {"gw_name": 'GW250114_082203', "duration":32, "unpack":True, "version": 2, "sample_rate":4, "center_at": 1420878141.2+off_center},
        {"gw_name": 'GW190521_074359', "duration":32, "unpack":True, "version": 2, "sample_rate":4, "center_at": 1242459857.4+off_center},
    ]
    event = events[event_idx]

    # Load strain data
    times, strain, t0_gps = DEPR_get_strain_data(**event, absolute_path=abs_path_to_use)

    # Compute marker to select the window
    marker = convert_gps_to_seconds(event["center_at"]-off_center, t0=t0_gps)

    # Compute power spectral density using Welch averaging
    f, ps, windows = calculate_welch_average(x=times, y=strain, L=3, output_on_full_harmonic_domain=True)

    # Loop through windows to find the one containing the event
    for current_window in windows:
        current_time, current_strain = current_window
        if current_time.min() < marker < current_time.max():
            # Whiten the strain in this window
            my_whitened = whiten(y=current_strain, amp=jnp.sqrt(ps))
            # Optionally bandpass (comment out if not needed)
            bandpass_bool = False
            if bandpass_bool:
                my_bandpassed = bandpass(x=current_time, y=my_whitened)
                print("BANDPASSING")
            else:
                print("NOT bandpassing")
            # actually no idea why was bandpassing and whitening here I was not using these variables
            # Solve for xi in frequency space
            current_strain_windowed = tukey_window_array(current_strain)
            xi_d_tilde = solve_data_equation_for_xi(data=current_strain_windowed, ps=ps)
            return xi_d_tilde, current_time, f

    raise ValueError("No window contains the event marker.")


In [15]:
# abs_path_to_use = "/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects/H-H1_GWOSC_4KHZ_R1-1242459842-32.hdf5"  # idx=3
abs_path_to_use = "/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects/H-H1_LOSC_4_V1-1126256640-4096.hdf5"  # idx=0
xi_d_tilde, times_window, freqs = get_xi_for_event(event_idx=0, abs_path_to_use=abs_path_to_use)

Start: Calculating welch average

Constructing 11 windows over which we average.

NOT bandpassing


In [16]:
whitened_xi = jnp.fft.ifft(xi_d_tilde).real

# plt.plot(times_window, whitened_xi)
# plt.show()

In [17]:
print("come here")
S_mat_inference, t_dual_inference, f_inference = Stress_re_debug(xi=whitened_xi, time=times_window, tukey_window_where_necessary=False)
S_mat_inference_tukey, t_dual_inference_tukey, f_inference_tukey = Stress_re_debug(xi=whitened_xi, time=times_window, tukey_window_where_necessary=True)

come here
not INVERSE FOURIER TRANSFORMING xi

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.0279826965413974e-20) 
not INVERSE FOURIER TRANSFORMING xi

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
NOT TUKEYING PHI, since it looks like in all cases its almost periodic...
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.6475845364670928e-22) 


In [39]:
visualize_stress(S_mat_inference, rows=f_inference, cols=t_dual_inference, smooth=True)
# visualize_stress(smooth_matrix(S_mat_inference, 5)**2, rows=f_inference, cols=t_dual_inference, smooth=False)
# visualize_stress(S_mat_inference_tukey, rows=f_inference_tukey, cols=t_dual_inference_tukey, smooth=True)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [26]:
# For plotting detection stat
S_mat_inference_shifted = np.fft.fftshift(S_mat_inference)
S_mat_inference_tukey_shifted = np.fft.fftshift(S_mat_inference_tukey)

In [35]:
X = smooth_matrix(np.fft.fftshift(S_mat_inference, axes=0), 5)**2
shifted_freqs = np.fft.fftshift(f_inference, axes=0)
where_DC = np.where(shifted_freqs == 0)[0][0]
Y = X[where_DC, :]
# plt.imshow(X.real, origin="lower")

In [41]:
plt.plot(t_dual_inference, Y.real)

In [40]:
# Write helper function

def detection_statistic(stress_matrix, plot=True, time_var=None, custom_ax=None, title=None, normalize=False, lb=""):
    """

    :param stress_matrix: 2D array, shape (n_f, n_t):   Output from stress_jft() function in standard DFT order.
    :param normalize: bool,                             If True, DC-line is divided by its max and the average is printed.

    Picks out the interference pattern on the DC-line of a Wigner-derived phase-space distribution through the following
    operations:

        1. Shifts from standard DFT order to DC-centered ordered along columns
        2. Smooths through Gaussian convolution
        3. Takes the absolute square. We call the resulting matrix 'smoothed Wigner power' (power => positive)
        4. Extracts DC-line from smoothed Wigner power

    :return dc_line,    The line corresponding to f=0 in the smoothed Wigner power
    :return SWP,        Smoothed Wigner power matrix in standard DFT order.

    """

    X = np.fft.fftshift(stress_matrix, axes=0)  # shift rows, corresponding to frequencies, such that f=0 is "in the middle" of the matrix. stress_matrix MUST be in standard DFT order (f=0 at the very bottom/top)
    SWP = smooth_matrix(X, smoothing_lvl=5).real**2
    where_dc = SWP.shape[0]//2
    dc_line = SWP[where_dc, :]

    if normalize:
        dc_line /= np.max(dc_line)
        print("Average of smoothed Wigner power's DC-line: ", np.average(dc_line))

    if plot:
        if time_var is None:
            raise ValueError("To plot, please provide time array")
        if custom_ax is None:
            _ = plt.figure()
            axis = plt.gca()
        else:
            axis = custom_ax
        axis.plot(time_var, dc_line, label=lb)
    return dc_line, np.fft.ifftshift(SWP, axes=0)

_ = detection_statistic(S_mat_inference, time_var=t_dual_inference)

In [25]:
import numpy as np


def plot_middle_slice(mat, time, smoothing_lvl=None, abs_square=False, title=None, normalize=False, plot=True, lb=""):
    """
    Plots the middle slice of a 2D matrix along the first axis (frequency-like axis).

    :param mat: 2D array, shape (n_f, n_t)
    :param time: 1D array, length n_t, corresponding to the second axis of mat
    :param smoothing_lvl: int or None, if given applies Gaussian smoothing via smooth_matrix
    :param abs_square: bool, if True, takes the squared absolute value of the matrix
    :param title: optional plot title
    """
    import copy
    mat_plot = copy.deepcopy(mat)

    # Optional smoothing
    if smoothing_lvl is not None:
        mat_plot = smooth_matrix(mat_plot, smoothing_lvl=smoothing_lvl, mode="gaussian")

    # Optional absolute square
    if abs_square:
        mat_plot = np.array(mat_plot.copy().real**2)

    # Take middle slice along first axis
    middle_idx = mat_plot.shape[0] // 2
    slice_middle = mat_plot[middle_idx, :]

    if normalize:
        slice_middle /= np.max(slice_middle)
        print("avg: ", np.average(slice_middle))

    if plot:
        # Plot
        plt.figure(figsize=(8,4))
        plt.plot(time, slice_middle, label=lb)
        plt.xlabel("Time")
        plt.ylabel("Amplitude")
        if title is not None:
            plt.title(title)
        plt.grid(True)
        plt.show()
        plt.legend()
    return slice_middle



In [31]:
slice_middle_1 = plot_middle_slice(S_mat_inference_shifted, times_window, smoothing_lvl=3, abs_square=True, normalize=False, plot=True, lb="OG")
# slice_middle_2 = plot_middle_slice(S_mat_inference_tukey_shifted, times_window, smoothing_lvl=3, abs_square=True, normalize=True, plot=True)
# slice_middle_2 = plot_middle_slice(S_mat_inference_tukey_shifted, t_dual_inference_tukey, smoothing_lvl=3, abs_square=True, normalize=False, plot=True, lb="noise reduced")

In [ ]:
# Select the time range
mask = (times_window < 15.5) | (times_window > 15.63)

# Apply mask to slices and time
t_sel = times_window[mask]
s1_sel = slice_middle_1[mask]
s2_sel = slice_middle_2[mask]

# Plot both slices
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(t_sel, s1_sel, label="Slice 1")
plt.plot(t_sel, s2_sel, label="Slice 2")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.title("Slices between 15.5 and 15.63")
plt.legend()
plt.grid(True)
plt.show()

# Compute and print mean values
print("Mean of Slice 1 in range:", np.mean(s1_sel))
print("Mean of Slice 2 in range:", np.mean(s2_sel))


In [ ]:
def pad_matrix(M, extent=0.1):
    """
    Pad a square matrix on all sides with its boundary values.

    Parameters
    ----------
    M : 2D square array
    extent : float
        Fraction of matrix size to pad on each side

    Returns
    -------
    M_padded : 2D array
        Padded matrix
    """
    M = np.asarray(M)
    if M.shape[0] != M.shape[1]:
        raise ValueError("M must be square")

    n = M.shape[0]
    pad = int(n * extent)

    print("Padding with ", pad, " elements, corresponding to ", np.round(100*pad/n,2), "% of array length")

    # top/bottom padding: repeat first/last row
    top = np.repeat(M[0:1, :], pad, axis=0)
    bottom = np.repeat(M[-1:, :], pad, axis=0)

    M_vert = np.vstack([top, M, bottom])

    # left/right padding: repeat first/last column
    left = np.repeat(M_vert[:, 0:1], pad, axis=1)
    right = np.repeat(M_vert[:, -1:], pad, axis=1)

    M_padded = np.hstack([left, M_vert, right])
    return M_padded

def smooth_zero_pad_core(M, extent=0.1):
    """
    Pad a square matrix with its boundary values and taper only the padding toward zero.
    Inner core remains absolutely untouched.

    Parameters
    ----------
    M : 2D square array
    extent : float
        Fraction of matrix size used as padding

    Returns
    -------
    M_smooth : 2D array
        Padded matrix with aggressively tapered edges
    """
    M = np.asarray(M)
    n = M.shape[0]
    pad = int(n * extent)
    if pad == 0:
        return M.copy()

    # Step 1: pad with boundary values
    M_pad = pad_matrix(M, extent=extent)
    N = M_pad.shape[0]

    # Step 2: create taper windows for padding region only
    wx = np.ones(N)
    wy = np.ones(N)

    # Left/right taper
    if pad > 0:
        wx[:pad] = 0.5 * (1 - np.cos(np.pi * np.linspace(0, 1, pad)))
        wx[-pad:] = 0.5 * (1 - np.cos(np.pi * np.linspace(1, 0, pad)))
        wy[:pad] = 0.5 * (1 - np.cos(np.pi * np.linspace(0, 1, pad)))
        wy[-pad:] = 0.5 * (1 - np.cos(np.pi * np.linspace(1, 0, pad)))

    # Step 3: make 2D window
    window2d = wy[:, None] * wx[None, :]

    # Step 4: preserve inner core
    inner_slice = slice(pad, N-pad)
    M_smooth = M_pad.copy()
    # multiply only padding regions
    # top
    M_smooth[:pad, :] *= window2d[:pad, :]
    # bottom
    M_smooth[-pad:, :] *= window2d[-pad:, :]
    # left
    M_smooth[pad:-pad, :pad] *= window2d[pad:-pad, :pad]
    # right
    M_smooth[pad:-pad, -pad:] *= window2d[pad:-pad, -pad:]

    col_diff, row_diff = boundary_differences(M_smooth)

    if np.any(np.round(np.sum(col_diff.real), 12) != 0) or \
       np.any(np.round(np.sum(col_diff.imag), 12) != 0) or \
       np.any(np.round(np.sum(row_diff.real), 12) != 0) or \
       np.any(np.round(np.sum(row_diff.imag), 12) != 0):
        raise ValueError("Boundary differences are nonzero; increase extent to reduce leakage.")

    return M_smooth


# Original matrix
# M = np.array([[1, 2, 3],
#               [4, 5, 6],
#               [7, 8, 9]], dtype=float)


# Apply smooth zero padding while keeping inner core untouched
# M_smooth = smooth_zero_pad_core(M, extent=0.7)
# print("Original M")
# print(M)
# print("turns to")
# print(M_smooth)


In [ ]:
def Stress_re_debug_custom_pad(xi, time, padding_extent=.1, supress_print=False, downsample=False, norm="ortho", tukey_window_where_necessary=False):
    """
    Implements S_ft, i.e. rows are frequencies and columns are times.

    See also nifty8 `Stress` function.

    :param xi: jnp.array        A field to calculate the wigner function for. Either of complex or real data type.
                                If complex, assumed to be in DFT standard order (DC first, then positives then negatives).
    :param time: jnp.array      The real-space time array at which xi (or its iFFT if complex) was sampled at.
    :param supress_print: bool, Print imaginary part diagonstics (Wigner function should be real).
    :return:
    """

    t0 = time[0]
    dt = time[1]-time[0]

    # extent = 1 + padding_extent
    N = len(xi) # * extent
    f = jnp.fft.fftfreq(N, d=dt)
    k = f.copy()
    df = f[1] - f[0]
    t = jnp.arange(N) / (N*df)  # dual time, equal to input time - time[0].
    T = N * dt

    FFT_physical = lambda x, ax=-1: jnp.fft.fft(x, norm=norm, axis=ax) * T / jnp.sqrt(N)
    iFFT_physical = lambda x, ax=-1: jnp.fft.ifft(x, norm=norm, axis=ax) * jnp.sqrt(N) / T

    if jnp.iscomplexobj(xi):
        print("INVERSE FOURIER TRANSFORMING xi")
        xi = iFFT_physical(xi)  # go to real space
    else:
        print("not INVERSE FOURIER TRANSFORMING xi")

    if downsample:
        step = 2
        xi = xi[::step]
        time = time[::step]

    if not supress_print:
        print("\nCalculating stress...")

    t_c = t[:, None]  # time cast
    k_c = k[None, :]  # shift frequencies cast
    xi_c = xi[:, None]  # xi values cast as rows

    if not supress_print:
        print("\t Calculating zeta plus")
    if tukey_window_where_necessary:
        # plot_boundary_differences(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c)
        # zeta_plus = tukey_window_matrix(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
        zeta_plus = smooth_zero_pad_core(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c, padding_extent) # domain = (time_space, h_space)
        plot_boundary_differences(zeta_plus)
    else:
        zeta_plus = jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta minus")

    if tukey_window_where_necessary:
        # plot_boundary_differences(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c)
        # zeta_minus = tukey_window_matrix(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
        zeta_minus = smooth_zero_pad_core(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c, padding_extent) # domain = (time_space, h_space)
        plot_boundary_differences(zeta_minus)
        # stop
    else:
        zeta_minus = jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c  # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta plus in Fourier space")
    tilde_zeta_plus = FFT_physical(zeta_plus, ax=0)

    if not supress_print:
        print("\t Calculating zeta minus in Fourier space")
    tilde_zeta_minus = FFT_physical(zeta_minus, ax=0)

    if not supress_print:
        print("\t Calculating Phi matrix")
    if tukey_window_where_necessary:
        # plot_boundary_differences(tilde_zeta_plus * tilde_zeta_minus.conj())
        # Phi = tukey_window_matrix(tilde_zeta_plus * tilde_zeta_minus.conj(), ax=1)  # domain = (h_space, h_space)
        print("NOT TUKEYING PHI, since it looks like in all cases its almost periodic...")
        Phi = tilde_zeta_plus * tilde_zeta_minus.conj()  # domain = (h_space, h_space)
        plot_boundary_differences(Phi)
    else:
        Phi = tilde_zeta_plus * tilde_zeta_minus.conj()  # domain = (h_space, h_space)

    if not supress_print:
        print("\t Inverse Fourier-Transforming columns of Phi matrix")
    S = iFFT_physical(Phi, ax=1)
    S.block_until_ready()

    if not supress_print:
        print("\t ... Done")
    if not supress_print:
        diagnostic = jnp.abs(jnp.mean(S.imag))
        tmp = float(diagnostic)
        if diagnostic < 1e-10:
            print(f"\u2714 Mean imaginary part of stress field is smaller than 1e-10 threshold ({diagnostic}) ")
        else:
            raise_warning(
                f"Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 ({diagnostic}).")

    if tukey_window_where_necessary:
        pad = int(len(xi) * padding_extent)
        N_pad = len(xi) + 2*pad
        f_pad = jnp.fft.fftfreq(N_pad, d=dt)
        df_pad = f[1] - f[0]
        t_pad = jnp.arange(N_pad) / (N_pad*df_pad)  # dual time, equal to input time - time[0].
        return S, t_pad+t0, f_pad
    else:
        return S, t+t0, f

In [ ]:
S_mat_nrt_tukey, t_dual_nrt_tukey, f_dual_nrt_tukey = Stress_re_debug_custom_pad(xi=nrt_strain_values, time=nrt_time_values, tukey_window_where_necessary=True,
                                                                padding_extent=0.01)

In [ ]:
S_mat_nrt, t_dual_nrt, f_nrt = Stress_re_debug_custom_pad(xi=nrt_strain_values, time=nrt_time_values, tukey_window_where_necessary=False)

In [ ]:
visualize_stress(S_mat_nrt_tukey, rows=f_dual_nrt_tukey, cols=t_dual_nrt_tukey, smooth=False)

In [ ]:
white_stress_real_res2, t_dual_res_2, f_dual_res_2 = Stress_re_debug_custom_pad(xi=xi_white_real_res2, time=time_res2, tukey_window_where_necessary=True, padding_extent=0.01)

In [ ]:
visualize_stress(white_stress_real_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [ ]:
S_mat_inference_tukey, t_dual_inference_tukey, f_inference_tukey = Stress_re_debug_custom_pad(xi=whitened_xi, time=times_window, tukey_window_where_necessary=True, padding_extent=0.01)

S_mat_inference, t_dual_inference, f_inference = Stress_re_debug_custom_pad(xi=whitened_xi, time=times_window, tukey_window_where_necessary=False)

In [ ]:
print(S_mat_inference_tukey.shape, t_dual_inference_tukey.shape)

In [ ]:
visualize_stress(S_mat_inference, rows=f_inference, cols=t_dual_inference, smooth=True, )#xlim=(14.45, 14.58), ylim=(-400,400))

In [ ]:
visualize_stress(S_mat_inference_tukey, rows=f_inference_tukey, cols=t_dual_inference_tukey, smooth=True, )#xlim=(14.45, 14.58), ylim=(-400,400))